In [ ]:
import os
from pprint import pprint

import torch

from src.utils import read_config_file, open_data, sample_data
from src.data import LyricDataset
from src.model import BERT_SongWriter, LyricsRemixer


In [ ]:
# Folder creation
os.makedirs("../model_out", exist_ok=True)

# Reading configuration file
cfg = read_config_file(config_path="./configs/config.yaml")
pprint(cfg)

# Paths setup
root_folder = "../"
print("Project Root Folder: ", root_folder)
cwd = os.getcwd()
print("Current work directory: ", cwd)
data_path = os.path.join(root_folder, cfg["paths"]["data"])
print("data path: ", data_path)
model_folder = os.path.join(root_folder, cfg["paths"]["model_files"])
print("model files folder: ", model_folder)
dataset_path = os.path.join(data_path, "raw/dataset_header_fullEnglish.csv")
print("dataset path: ", dataset_path)

# Device setup
device = torch.device(cfg["project"]["device"])
print(f"device used: {device}")


CKPT = f"epoch={4}-step={5000}.ckpt"                # >>| Modifify the ckpt name you placed inside the ../model folder |<<

In [ ]:
N_SAMPLES = 1000
N_ALTERNATIVES = 50

gold_txt_lines = open_data(path=os.path.join(data_path,"raw/txt_lines.txt"), type="lines")
gold_txt_lines_pairs = open_data(path=os.path.join(model_folder, "txt_lines_pairs.txt"), type="lines-pairs")
print(f"Length of gold text-lines: {len(gold_txt_lines)}")
print(f"Length of gold text-lines pairs: {len(gold_txt_lines_pairs)}")

ALL_IN_DATA, ALL_IN_GROUND_TRUTH = sample_data(txt_lines=gold_txt_lines, txt_lines_pair=gold_txt_lines_pairs,
                                               n_samples=None, n_alternatives=N_ALTERNATIVES)


test_txt_lines = open_data(path=os.path.join(data_path, "test_txt_lines.txt"), type="lines")
print(f"Length test_txt_lines : {len(test_txt_lines)}")

test_in_data, test_ground_truth = sample_data(txt_lines=test_txt_lines, txt_lines_pair=gold_txt_lines_pairs,
                                              n_samples=N_SAMPLES, n_alternatives=N_ALTERNATIVES)


ckpt_path = os.path.join(model_folder, CKPT)
model = BERT_SongWriter.load_from_checkpoint(ckpt_path, device=device)
print("-- Model Loaded --")

test_lyric_dataset = LyricDataset(in_data=test_in_data, gold_data=test_ground_truth,
                                  tokenizer=model.tokenizer)

## Lyric Generation

In [ ]:
lyric_remixer = LyricsRemixer(model=model, device=device)

In [ ]:
new_song, new_gold_song = lyric_remixer.compose_new_song(test_in=test_in_data,
                                                         lines_dict=ALL_IN_DATA, truth_lines_dict=ALL_IN_GROUND_TRUTH,
                                                         song_length=20, limit_candidate=30, device=device)
# print("---- New Song Lyric ----"); print("\n")
# pprint(new_song)

# Abstractive Summarization: Song Title Creation and Evaluation
song_title, new_song = lyric_remixer.create_song_title(song=new_song,
                                                       min_l=1, max_l=3)
print(f"\n-- Final Song --\n\
      {song_title.upper()}\n\
      {new_song}")
lyric_remixer.judge_created_title(song=new_song, title=song_title)
with open(file="../model_out/song_writing1.txt", mode="a") as file:
    file.writelines(f"\n{song_title.upper()}\n{new_song}")